# Cross-Entropy and the Softmax Gradient: One Loss for Every Classifier

The runnable code of [Cross-Entropy and the Softmax Gradient: One Loss for Every Classifier](https://residual-stream.lvtan71.workers.dev/posts/softmax-and-cross-entropy/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/softmax-and-cross-entropy.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "softmax-and-cross-entropy"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/softmax-and-cross-entropy/router4.py', 'shared/features-overfitting-regularization/router_data.py', 'shared/softmax-and-cross-entropy/softmax_basics.py', 'data/router-questions.csv']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## From one sigmoid to four routes

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/softmax-and-cross-entropy/#from-one-sigmoid-to-four-routes)

In [ ]:
import numpy as np
from router_data import split
from softmax_basics import softmax
from router4 import four_way, train

def one_vs_rest(X, y, lam=0.003, steps=4000, eta=0.5):
    """Four independent sigmoid classifiers, one per route."""
    W, b = np.zeros((X.shape[1], 4)), np.zeros(4)
    Y = np.eye(4)[y]
    for _ in range(steps):
        P = 1 / (1 + np.exp(-(X @ W + b)))
        G = (P - Y) / len(y)                            # the same p - y, per column
        W -= eta * (X.T @ G + lam * W)
        b -= eta * G.sum(axis=0)
    return W, b

ce = lambda P, y: -np.log(P[np.arange(len(y)), y]).mean()
rows = []
for seed in range(20):
    ((Xtr, ytr), (Xva, yva), _), _ = four_way(*split(seed))
    Ws, bs = train(Xtr, ytr, lam=0.003)
    Wo, bo = one_vs_rest(Xtr, ytr)
    P = softmax(Xva @ Ws + bs)
    S = 1 / (1 + np.exp(-(Xva @ Wo + bo)))
    rows.append(((P.argmax(1) == yva).mean(), (S.argmax(1) == yva).mean(),
                 np.abs(S.sum(1) - 1).mean(), (S.max(1) < 0.5).mean(), (P.max(1) < 0.5).mean(),
                 ce(P, yva), ce(S / S.sum(1, keepdims=True), yva)))
r = np.array(rows)
se = lambda d: f"{d.mean():+.3f} ± {d.std(ddof=1) * np.sqrt(1 / 20 + 40 / 120):.3f}"
print(f"val accuracy: softmax {r[:, 0].mean():.3f}, one-vs-rest {r[:, 1].mean():.3f} "
      f"(difference {se(r[:, 1] - r[:, 0])})")
print(f"one-vs-rest scores: sum is off from 1 by {r[:, 2].mean():.2f} on average; no route above 0.5 "
      f"on {r[:, 3].mean():.0%} of questions (softmax: {r[:, 4].mean():.0%})")
print(f"val CE: softmax {r[:, 5].mean():.3f}, one-vs-rest rescaled to sum to 1 {r[:, 6].mean():.3f} "
      f"(difference {se(r[:, 6] - r[:, 5])})")

The post printed:

```text
val accuracy: softmax 0.621, one-vs-rest 0.624 (difference +0.003 ± 0.016)
one-vs-rest scores: sum is off from 1 by 0.19 on average; no route above 0.5 on 53% of questions (softmax: 33%)
val CE: softmax 0.891, one-vs-rest rescaled to sum to 1 0.885 (difference -0.006 ± 0.011)
```

## How surprised should the model be?

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/softmax-and-cross-entropy/#how-surprised-should-the-model-be)

In [ ]:
import numpy as np

def H(p):                  # entropy, in nats
    p = np.asarray(p, float)
    return -(p[p > 0] * np.log(p[p > 0])).sum()

def CE(p, q):              # cross-entropy of model q on true distribution p
    p, q = np.asarray(p, float), np.asarray(q, float)
    return -(p[p > 0] * np.log(q[p > 0])).sum()

truth = [0.7, 0.1, 0.1, 0.1]     # what the answer really is, for some question
for name, q in {"matches the truth": truth,
                "uniform guess":     [0.25] * 4,
                "overconfident":     [0.97, 0.01, 0.01, 0.01],
                "confidently wrong": [0.01, 0.97, 0.01, 0.01]}.items():
    ce = CE(truth, q)
    print(f"{name:18s} CE {ce:.3f} = H {H(truth):.3f} + KL {ce - H(truth):.3f}   "
          f"perplexity {np.exp(ce):.2f}")

print(f"\nuniform over 4 routes: ln 4 = {np.log(4):.3f} nats; "
      f"over a 128,256-token vocabulary: {np.log(128256):.2f} nats")
print(f"val CE 0.874 -> perplexity {np.exp(0.874):.2f}")

# Label smoothing: the best a model can do is to predict the smoothed target,
# which caps the gap between the right logit and the others.
for eps in (0.1, 0.01):
    top, rest = 1 - eps + eps / 4, eps / 4
    print(f"smoothing {eps}: target ({top:.4f}, {rest:.4f} x3), logit gap ln({top:.4f}/{rest:.4f}) "
          f"= {np.log(top / rest):.2f}, loss floor {H([top, rest, rest, rest]):.3f}")

The post printed:

```text
matches the truth  CE 0.940 = H 0.940 + KL 0.000   perplexity 2.56
uniform guess      CE 1.386 = H 0.940 + KL 0.446   perplexity 4.00
overconfident      CE 1.403 = H 0.940 + KL 0.462   perplexity 4.07
confidently wrong  CE 4.148 = H 0.940 + KL 3.207   perplexity 63.29

uniform over 4 routes: ln 4 = 1.386 nats; over a 128,256-token vocabulary: 11.76 nats
val CE 0.874 -> perplexity 2.40
smoothing 0.1: target (0.9250, 0.0250 x3), logit gap ln(0.9250/0.0250) = 3.61, loss floor 0.349
smoothing 0.01: target (0.9925, 0.0025 x3), logit gap ln(0.9925/0.0025) = 5.98, loss floor 0.052
```

## The gradient is still prediction minus target

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/softmax-and-cross-entropy/#the-gradient-is-still-prediction-minus-target)

In [ ]:
%%writefile softmax_basics.py
import numpy as np

def softmax(Z):
    """Row-wise softmax, shifted by the row max so exp never overflows."""
    Z = Z - Z.max(axis=-1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=-1, keepdims=True)

def cross_entropy(Z, y):
    """Mean of -log p[y], computed from logits with log-sum-exp."""
    Z = Z - Z.max(axis=-1, keepdims=True)
    log_p = Z - np.log(np.exp(Z).sum(axis=-1, keepdims=True))
    return -log_p[np.arange(len(y)), y].mean()

if __name__ == "__main__":
    # The gradient of cross-entropy w.r.t. the logits is p - onehot(y).
    rng = np.random.default_rng(0)
    z, target = rng.normal(size=(1, 4)), np.array([2])
    analytic = softmax(z)[0] - np.eye(4)[2]
    numeric = np.array([(cross_entropy(z + e, target) - cross_entropy(z - e, target)) / 2e-6
                        for e in np.eye(4)[:, None, :] * 1e-6])
    print(f"p - y:  {np.round(analytic, 4)}")
    print(f"largest gap to finite differences: {np.abs(numeric - analytic).max():.1e}")

    # Why the shift matters.
    z = np.array([[1000.0, 999.0, 0.0, -5.0]])
    with np.errstate(over="ignore", invalid="ignore"):
        naive = np.exp(z) / np.exp(z).sum()
    print(f"naive softmax: {naive[0]}   shifted: {np.round(softmax(z)[0], 4)}")
    show = lambda v: "[" + " ".join(f"{x:.2f}" for x in v[0]) + "]"
    with np.errstate(divide="ignore"):
        print(f"log(softmax(z)): {show(np.log(softmax(z)))}   "
              f"log-sum-exp: {show(z - z.max() - np.log(np.exp(z - z.max()).sum()))}")

In [ ]:
%run softmax_basics.py

The post printed:

```text
p - y:  [ 0.226   0.1746 -0.6219  0.2213]
largest gap to finite differences: 8.5e-11
naive softmax: [nan nan  0.  0.]   shifted: [0.7311 0.2689 0.     0.    ]
log(softmax(z)): [-0.31 -1.31 -inf -inf]   log-sum-exp: [-0.31 -1.31 -1000.31 -1005.31]
```

## Four routes

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/softmax-and-cross-entropy/#four-routes)

In [ ]:
%%writefile router4.py
import numpy as np
from router_data import featurize, ROUTES
from softmax_basics import softmax

def four_way(train, *others):
    """Bag of words from "Features and Overfitting", with the route index (0-3) as the label."""
    sets, vocab = featurize(train, *others)
    return [(X, np.array([ROUTES.index(r["route"]) for r in rs]))
            for (X, _), rs in zip(sets, (train, *others))], vocab

def train(X, y, lam=0.0, smooth=0.0, steps=4000, eta=0.5, K=4):
    W, b = np.zeros((X.shape[1], K)), np.zeros(K)
    Y = np.eye(K)[y] * (1 - smooth) + smooth / K        # label smoothing
    for _ in range(steps):
        G = (softmax(X @ W + b) - Y) / len(y)           # p - y, per example
        W -= eta * (X.T @ G + lam * W)
        b -= eta * G.sum(axis=0)
    return W, b

def top_label_ece(P, y, bins=5):
    conf, right = P.max(axis=1), P.argmax(axis=1) == y
    which = np.clip((conf * bins).astype(int), 0, bins - 1)
    return sum((which == k).mean() * abs(conf[which == k].mean() - right[which == k].mean())
               for k in range(bins) if (which == k).any())

In [ ]:
import numpy as np
from router_data import split, ROUTES
from softmax_basics import cross_entropy, softmax
from router4 import four_way, train, top_label_ece

CONFIGS = {"L2 0.003": (0.003, 0.0), "no penalty": (0.0, 0.0),
           "smoothing 0.1": (0.0, 0.1), "smoothing 0.1 + L2 0.003": (0.003, 0.1)}
val, test = {c: [] for c in CONFIGS}, {c: [] for c in CONFIGS}
for seed in range(20):
    ((Xtr, ytr), (Xva, yva), (Xte, yte)), _ = four_way(*split(seed))
    for name, (lam, smooth) in CONFIGS.items():
        W, b = train(Xtr, ytr, lam, smooth)
        P = softmax(Xva @ W + b)
        val[name].append((cross_entropy(Xva @ W + b, yva), (P.argmax(1) == yva).mean(),
                          top_label_ece(P, yva)))
        test[name].append((softmax(Xte @ W + b).argmax(1), yte, cross_entropy(Xte @ W + b, yte)))

base = np.array(val["L2 0.003"])
corrected = np.sqrt(1 / 20 + 40 / 120)                  # Nadeau-Bengio, "Features and Overfitting"
vs = lambda d: f"{d.mean():+.3f} ± {d.std(ddof=1) * corrected:.3f}" if d.any() else ""
print(f"{'':25s} {'val CE':>7s} {'vs L2':>15s} {'val acc':>8s} {'vs L2':>15s} {'ECE':>6s} {'vs L2':>15s}")
for name, rows in val.items():
    r = np.array(rows)
    d = r - base                                         # paired, split by split
    print(f"{name:25s} {r[:, 0].mean():7.3f} {vs(d[:, 0]):>15s} {r[:, 1].mean():8.3f} "
          f"{vs(d[:, 1]):>15s} {r[:, 2].mean():6.3f} {vs(d[:, 2]):>15s}")
both, smooth = np.array(val["smoothing 0.1 + L2 0.003"]), np.array(val["smoothing 0.1"])
print(f"smoothing + L2 vs smoothing alone: val CE {vs(both[:, 0] - smooth[:, 0])}")

best = min(val, key=lambda c: np.mean([row[0] for row in val[c]]))   # chosen on validation
conf = np.zeros((4, 4), dtype=int)
for pred, true, _ in test[best]:
    for t, p in zip(true, pred):
        conf[t, p] += 1
test_ce = np.mean([ce for *_, ce in test[best]])
print(f"\nchosen on validation CE: {best}. test CE {test_ce:.3f} (perplexity {np.exp(test_ce):.2f})")
print("test confusion, pooled over the 20 splits (rows = true route):")
print(f"{'':14s}" + "".join(f"{r:>13s}" for r in ROUTES))
for i, r in enumerate(ROUTES):
    print(f"{r:14s}" + "".join(f"{v:13d}" for v in conf[i]))
precision, recall = conf.diagonal() / conf.sum(0), conf.diagonal() / conf.sum(1)
f1 = 2 * precision * recall / (precision + recall)
for r, p, q, f in zip(ROUTES, precision, recall, f1):
    print(f"{r:14s} precision {p:.3f}  recall {q:.3f}  F1 {f:.3f}")
print(f"test accuracy {conf.trace() / conf.sum():.4f}; mean recall {recall.mean():.4f}; "
      f"macro-F1 {f1.mean():.4f}")

The post printed:

```text
                           val CE           vs L2  val acc           vs L2    ECE           vs L2
L2 0.003                    0.891                    0.621                  0.096                
no penalty                  0.976  +0.085 ± 0.054    0.633  +0.011 ± 0.021  0.154  +0.059 ± 0.033
smoothing 0.1               0.874  -0.017 ± 0.022    0.655  +0.034 ± 0.029  0.106  +0.010 ± 0.033
smoothing 0.1 + L2 0.003    0.907  +0.016 ± 0.014    0.628  +0.006 ± 0.013  0.125  +0.029 ± 0.031
smoothing + L2 vs smoothing alone: val CE +0.033 ± 0.018

chosen on validation CE: smoothing 0.1. test CE 0.848 (perplexity 2.34)
test confusion, pooled over the 20 splits (rows = true route):
                    serving architecture       agents  foundations
serving                 120           29           22           29
architecture             19          113           12           56
agents                   10           10          153           27
foundations              17           19           20          144
serving        precision 0.723  recall 0.600  F1 0.656
architecture   precision 0.661  recall 0.565  F1 0.609
agents         precision 0.739  recall 0.765  F1 0.752
foundations    precision 0.562  recall 0.720  F1 0.632
test accuracy 0.6625; mean recall 0.6625; macro-F1 0.6621
```

In [ ]:
import numpy as np
from router_data import split
from softmax_basics import softmax, cross_entropy
from router4 import four_way, train

# 1. Is 4,000 steps enough, or is it quietly early-stopping some models?
# 2. What does each model look like on its own training data?
CONFIGS = {"L2 0.003": (0.003, 0.0), "no penalty": (0.0, 0.0),
           "smoothing 0.1": (0.0, 0.1), "smoothing 0.1 + L2 0.003": (0.003, 0.1)}
rows = {}
for seed in range(20):
    ((Xtr, ytr), (Xva, yva), _), _ = four_way(*split(seed))
    for name, (lam, smooth) in CONFIGS.items():
        for steps in (4000, 16000):
            W, b = train(Xtr, ytr, lam, smooth, steps=steps)
            P = softmax(Xva @ W + b)
            rows.setdefault((name, steps), []).append((
                cross_entropy(Xva @ W + b, yva), np.linalg.norm(W),
                cross_entropy(Xtr @ W + b, ytr),              # hard labels, no penalty
                P.max(1).mean(), (P.argmax(1) == yva).mean()))
corrected = np.sqrt(1 / 20 + 40 / 120)
base = {s: np.array(rows[("L2 0.003", s)])[:, 0] for s in (4000, 16000)}
print(f"{'':25s} {'steps':>6s} {'val CE':>7s} {'vs L2':>15s} {'|W|':>6s} {'train CE':>9s} "
      f"{'mean top p':>11s} {'val acc':>8s}")
for (name, steps), r in rows.items():
    r = np.array(r)
    d = r[:, 0] - base[steps]
    vs = f"{d.mean():+.3f} ± {d.std(ddof=1) * corrected:.3f}" if d.any() else ""
    print(f"{name:25s} {steps:6,d} {r[:, 0].mean():7.3f} {vs:>15s} {r[:, 1].mean():6.1f} "
          f"{r[:, 2].mean():9.3f} {r[:, 3].mean():11.3f} {r[:, 4].mean():8.3f}")

The post printed:

```text
                           steps  val CE           vs L2    |W|  train CE  mean top p  val acc
L2 0.003                   4,000   0.891                   10.6     0.131       0.622    0.621
L2 0.003                  16,000   0.891                   10.6     0.131       0.622    0.621
no penalty                 4,000   0.976  +0.085 ± 0.054   21.7     0.008       0.776    0.633
no penalty                16,000   1.068  +0.177 ± 0.078   27.3     0.002       0.815    0.630
smoothing 0.1              4,000   0.874  -0.017 ± 0.022   14.1     0.080       0.616    0.655
smoothing 0.1             16,000   0.876  -0.015 ± 0.025   14.7     0.078       0.613    0.661
smoothing 0.1 + L2 0.003   4,000   0.907  +0.016 ± 0.014    9.0     0.200       0.563    0.628
smoothing 0.1 + L2 0.003  16,000   0.907  +0.016 ± 0.014    9.0     0.200       0.563    0.628
```

## Reading a 4×4 confusion matrix

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/softmax-and-cross-entropy/#reading-a-44-confusion-matrix)

In [ ]:
import numpy as np
from router_data import split, tokens, ROUTES
from softmax_basics import softmax
from router4 import four_way, train

tr, va, te = split(0)
((Xtr, ytr), (Xte, yte)), vocab = four_way(tr, te)
W, b = train(Xtr, ytr, smooth=0.1)
for k, route in enumerate(ROUTES):
    top = np.argsort(-W[:, k])[:6]
    print(f"{route:13s} " + ", ".join(f"{vocab[j]} {W[j, k]:+.2f}" for j in top))
print(f"\nbias b = {np.round(b, 2)}, softmax(b) = {np.round(softmax(b), 2)}")
P = softmax(Xte @ W + b)
for i in np.where((yte == 1) & (P.argmax(1) == 3))[0]:
    words = tokens(te[i]["question"])
    print(f"\narchitecture -> foundations: {te[i]['question']}")
    print("  never seen in training: " + ", ".join(t for t in words if t not in vocab))
    for t in dict.fromkeys(t for t in words if t in vocab):
        print(f"  {t:5s} weights {np.round(W[vocab.index(t)], 2)}")
    print(f"  words alone: p = {np.round(softmax(Xte[i] @ W), 2)}   with the bias: p = {np.round(P[i], 2)}")

The post printed:

```text
serving       vllm +1.11, batching +1.11, token +0.98, paged +0.96, to +0.95, quantization +0.92
architecture  attention +1.92, unembedding +1.61, residual +1.59, stream +1.29, transformers +1.21, layer +1.17
agents        agent +1.83, prompt +1.63, context +1.47, calling +1.38, rot +1.33, retrieval +1.22
foundations   backpropagation +1.74, confusion +1.29, learning +1.23, gradients +1.23, sigmoid +1.12, optimizer +1.09

bias b = [ 0.13 -0.17 -0.17  0.21], softmax(b) = [0.28 0.21 0.21 0.3 ]

architecture -> foundations: How does rotary position embedding encode distance?
  never seen in training: rotary, position, embedding, encode, distance
  how   weights [-0.7   0.19  0.27  0.24]
  does  weights [ 0.43 -0.2  -0.01 -0.23]
  words alone: p = [0.19 0.24 0.32 0.25]   with the bias: p = [0.22 0.21 0.27 0.31]

architecture -> foundations: How is the embedding table trained?
  never seen in training: embedding, table, trained
  how   weights [-0.7   0.19  0.27  0.24]
  is    weights [ 0.04 -0.24 -0.15  0.34]
  the   weights [ 0.02  0.57 -0.43 -0.16]
  words alone: p = [0.12 0.38 0.16 0.34]   with the bias: p = [0.13 0.31 0.14 0.41]
```

In [ ]:
import numpy as np
from router_data import english, split, tokens, ROUTES
from softmax_basics import softmax
from router4 import four_way, train

unsure, sent_to, all_to, unseen = [], np.zeros(4), np.zeros(4), {True: [], False: []}
to_found, moved, right = 0, 0, {True: [], False: []}
for seed in range(20):
    tr, va, te = split(seed)
    ((Xtr, ytr), (Xte, yte)), vocab = four_way(tr, te)
    W, b = train(Xtr, ytr, smooth=0.1)
    P = softmax(Xte @ W + b)
    low = P.max(1) < 0.4                                 # the router is barely guessing
    unsure.append(low.mean())
    sent_to += np.bincount(P[low].argmax(1), minlength=4)
    all_to += np.bincount(P.argmax(1), minlength=4)
    for i, r in enumerate(te):
        unseen[bool(low[i])].append(np.mean([t not in vocab for t in tokens(r["question"])]))
        right[bool(low[i])].append(P[i].argmax() == yte[i])
    # Would they still go to foundations if no route had a head start from its bias?
    found = low & (P.argmax(1) == 3)
    to_found += found.sum()
    moved += (softmax(Xte[found] @ W + b.mean()).argmax(1) != 3).sum()
print(f"test questions with top probability < 0.4: {np.mean(unsure):.0%}, "
      f"right {np.mean(right[True]):.0%} of the time (the rest: {np.mean(right[False]):.0%})")
print("where they go:   " + ", ".join(f"{r} {n / sent_to.sum():.0%}" for r, n in zip(ROUTES, sent_to)))
print("all predictions: " + ", ".join(f"{r} {n / all_to.sum():.0%}" for r, n in zip(ROUTES, all_to)))
print(f"share of words never seen in training: unsure {np.mean(unseen[True]):.0%}, "
      f"the rest {np.mean(unseen[False]):.0%}")
print(f"of the {to_found} unsure questions sent to foundations, {moved} go elsewhere with an even bias")
for w in ["what", "is"]:
    print(f"questions containing '{w}': " + ", ".join(
        f"{r} {sum(w in tokens(q['question']) for q in english if q['route'] == r)}" for r in ROUTES))

The post printed:

```text
test questions with top probability < 0.4: 16%, right 40% of the time (the rest: 71%)
where they go:   serving 18%, architecture 23%, agents 22%, foundations 37%
all predictions: serving 21%, architecture 21%, agents 26%, foundations 32%
share of words never seen in training: unsure 33%, the rest 26%
of the 47 unsure questions sent to foundations, 24 go elsewhere with an even bias
questions containing 'what': serving 11, architecture 19, agents 16, foundations 28
questions containing 'is': serving 13, architecture 17, agents 14, foundations 28
```

## At language-model scale

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/softmax-and-cross-entropy/#at-language-model-scale)

In [ ]:
# Memory for the logits of one training batch, at Llama-3-8B's vocabulary.
V, D = 128_256, 4096
for B, T in [(1, 8192), (8, 8192)]:
    n = B * T                                            # token positions
    logits = n * V * 4                                   # FP32 logits
    print(f"B={B}, T={T}: {n:,} positions x {V:,} = {n * V / 1e9:.2f} B logits; "
          f"FP32 {logits / 2**30:.1f} GiB, and the same again for their gradient")
print(f"the hidden states that produce them, BF16: {8 * 8192 * D * 2 / 2**30:.2f} GiB")
print(f"with activation checkpointing, the saved input of each of 32 layers: "
      f"{32 * 8 * 8192 * D * 2 / 2**30:.0f} GiB")

The post printed:

```text
B=1, T=8192: 8,192 positions x 128,256 = 1.05 B logits; FP32 3.9 GiB, and the same again for their gradient
B=8, T=8192: 65,536 positions x 128,256 = 8.41 B logits; FP32 31.3 GiB, and the same again for their gradient
the hidden states that produce them, BF16: 0.50 GiB
with activation checkpointing, the saved input of each of 32 layers: 16 GiB
```